# Similar-case search (retrieval)

Indexes every historical case summary with TF-IDF and cosine **nearest neighbours**, keeping each case's type, category, priority, team, risk, date and resolution fields. At triage the app looks up the most similar past cases and shows how they were handled. This is retrieval, not a classifier, so it has no held-out metrics.

In [ ]:
import sys, os, warnings; warnings.filterwarnings("ignore")
search = os.getcwd()
while search != os.path.dirname(search) and not os.path.exists(os.path.join(search, "case_pipeline.py")):
    search = os.path.dirname(search)
sys.path.insert(0, search)
import numpy as np, pandas as pd
import case_pipeline as P   # shared loading / splits / features / evaluation

In [ ]:
df = P.load_clean(merge_team_aliases=True)

# Fields kept for each reference case so the app can show a rich similar-cases table.
REFERENCE_FIELDS = ["case_id", "case_summary", "case_type", "category", "priority",
                    "assigned_team", "escalated", "created_at",
                    "resolution_code", "resolution_time_hours", "status"]

retriever = P.fit_retriever(df, REFERENCE_FIELDS, text_col="case_summary", n_neighbors=10)

# Sanity check: nearest cases to an example query.
example = P.query_similar(retriever, "cannot export the monthly VAT invoice to Xero", k=5)
example[["case_summary", "case_type", "assigned_team", "priority", "similarity"]]

In [ ]:
P.save_bundle("similar_cases", {
    "retriever": retriever, "kind": "retrieval", "label": "Similar cases",
    "meta": {"approach": "TF-IDF + cosine nearest neighbours", "inputs": "summary only",
             "n_cases": int(len(df)),
             "note": "shows how similar historical cases were resolved"}})